# Цели проекта USCIS I-140 Analysis

## Общая цель

Цель проекта — собрать, нормализовать и проанализировать опубликованные USCIS агрегированные данные по I-140 и связанным отчетам за период FY2017-FY2025 включительно, чтобы оценить:

- как менялась нагрузка на USCIS по employment-based кейсам;
- как менялось количество заявок, одобрений, отказов и pending backlog;
- как менялись approval rate и denial rate по категориям кейсов;
- какие категории росли быстрее всего, в первую очередь EB1A / E11;
- как ведут себя свежие когорты, где значительная часть кейсов еще pending;
- во что исторически превращался pending: approval, denial или сохранение pending;
- какой актуальный ожидаемый процент одобрений и отказов можно оценить по категориям на основании исторического тренда.

Проект должен поддерживать регулярный пересчет по мере появления новых данных USCIS.

## Основной фокус

Главный интерес проекта — employment-based I-140 категории, особенно:

- EB1;
- EB1A / E11;
- EB2;
- NIW;
- EB3;
- подкатегории E12, E13, E21, E31, E32, EW3.

Отдельно важна идея snapshot/cohort analysis: свежие данные USCIS часто неполные, потому что значительная часть кейсов остается pending. Поэтому нужно анализировать не только “approval rate сейчас”, но и то, как старые когорты дозревали со временем.

## Ожидаемые результаты

Итоговый проект должен дать:

- чистую локальную PostgreSQL БД с исходниками, справочниками и fact tables;
- portable CSV-export для запуска анализа без PostgreSQL;
- pandas/Jupyter workflow для ручного исследования данных;
- набор sanity checks по качеству данных;
- графики квартальной и годовой динамики;
- аналитику по категориям EB1/EB2/EB3/NIW/EB1A;
- аналитику по странам, где данные это позволяют;
- cohort/snapshot analysis pending backlog;
- модель или набор правил для прогноза итогового approval/denial rate свежих когорт;
- итоговое заключение с ограничениями, допущениями и интерпретацией результатов;
- дашборд для визуального мониторинга ключевых метрик.

## Этап 1. Проверка и первичное знакомство с данными

Цель этапа — понять, что реально лежит в БД, где покрытие полное, а где есть ограничения.

Задачи:

- Посчитать количество строк в основных таблицах: `source_files`, `i140_status_counts`, `form_status_counts`, `forms`, `countries`, `case_statuses`, `preference_categories`.
- Проверить диапазон данных: минимальный и максимальный `report_fiscal_year`, доступные `report_quarter`, доступные `snapshot_date`.
- Посчитать количество source files по каждому `report_family`: `all_forms`, `fy_quarter_status`, `preference_country`, `country_category`.
- Проверить, какие годы и кварталы покрыты полностью, особенно Q1-Q4 для FY2017-FY2025.
- Проверить список стран в `uscis_dim.countries` и количество фактов по каждой стране.
- Проверить, какие категории I-140 реально есть в фактах: EB1, E11, E12, E13, EB2, E21, NIW, EB3, E31, E32, EW3, OTHER_UNKNOWN.
- Найти дубликаты на уровне логического ключа: source file, category, country, status, cohort fiscal year, cohort quarter, snapshot date.
- Проверить пропуски в ключевых колонках: `category_id`, `country_id`, `status_id`, `cohort_fiscal_year`, `snapshot_date`, `count_value`.
- Проверить отрицательные, нулевые и подозрительно большие значения `count_value`.
- Найти source files, которые дали больше всего строк фактов, и понять, являются ли они страновыми/категорийными отчетами.

## Этап 2. Простая квартальная динамика I-140 по all-forms

Для этого этапа основной источник — `uscis_fact.form_status_counts`, потому что он дает наиболее стабильную квартальную серию для I-140 в целом.

Задачи:

- Построить квартальную динамику `received` по I-140 за FY2017-FY2025.
- Построить квартальную динамику `approved`, `denied`, `pending` по I-140.
- Построить line chart: `received`, `approved`, `denied` по кварталам.
- Построить отдельный график `pending`, потому что масштаб backlog может быть другим.
- Посчитать годовые totals по I-140: `received`, `approved`, `denied`, `pending`.
- Построить годовой bar chart по `received`.
- Построить stacked bar chart по годам: `approved`, `denied`, `pending`.
- Найти квартал с максимальным количеством `received`.
- Найти квартал с максимальным количеством `denied`.
- Найти квартал с максимальным `pending backlog`.
- Посчитать quarter-over-quarter change для `received`: абсолютное и процентное изменение.
- Посчитать year-over-year change для `received` по одинаковым кварталам.
- Проверить сезонность: среднее количество `received` по Q1, Q2, Q3, Q4.
- Построить heatmap fiscal year x quarter для `received`.
- Построить heatmap fiscal year x quarter для `denied`.
- Построить heatmap fiscal year x quarter для `pending`.

## Этап 3. Approval, denial и pending rates

Цель этапа — увидеть разные способы расчета rates и не смешивать “received basis” с “decided basis”.

Задачи:

- Посчитать `approved / received` по кварталам для I-140.
- Посчитать `denied / received` по кварталам для I-140.
- Посчитать `pending / received` по кварталам для I-140.
- Посчитать `approved / (approved + denied)` по кварталам.
- Сравнить `approved / received` и `approved / (approved + denied)`.
- Построить line chart approval rate на received basis.
- Построить line chart approval rate на decided basis.
- Построить line chart pending share.
- Найти кварталы, где pending share превышает 50%, 70%, 90%.
- Найти кварталы, где denial rate на decided basis был максимальным.
- Сравнить FY2024 и FY2025 по `received`, `approved`, `denied`, `pending`, approval rate, denial rate, pending share.
- Построить rolling average для `received` на 4 квартала.
- Построить rolling average для denial rate на 4 квартала.
- Проверить, не ломают ли свежие кварталы общую картину из-за pending, отдельно сравнив FY2017-FY2022 и FY2023-FY2025.

## Этап 4. Аналитика по категориям EB1, EB2, EB3, NIW и EB1A

Основные источники — `uscis_fact.i140_status_counts` и `uscis_mart.i140_rates_by_snapshot`.

Задачи:

- Посчитать `received` по годам для EB1, EB2, EB3.
- Посчитать `received` по годам для E11, E12, E13, E21, NIW, E31, E32, EW3.
- Построить line chart `received` по EB1, EB2, EB3.
- Построить line chart `received` по NIW.
- Построить отдельный line chart для EB1A / E11.
- Посчитать долю NIW внутри EB2: `NIW received / EB2 received`.
- Посчитать долю E11 внутри EB1.
- Посчитать долю E13 внутри EB1.
- Посчитать долю EW3 внутри EB3.
- Построить stacked bar chart по EB2: E21 vs NIW.
- Построить stacked bar chart по EB1: E11, E12, E13.
- Построить stacked bar chart по EB3: E31, E32, EW3.
- Найти категорию с самым быстрым ростом `received`.
- Найти категорию с самым сильным падением `received`.
- Посчитать CAGR для `received` по категориям.
- Посчитать approval rate на decided basis по категориям.
- Посчитать denial rate на decided basis по категориям.
- Посчитать pending share по категориям.
- Сравнить NIW с остальным EB2.
- Сравнить E11 с остальным EB1.
- Сравнить EB1, EB2, EB3 по pending share в свежих отчетах.
- Найти категории, где pending особенно долго сохраняется по старым cohort periods.

## Этап 5. Аналитика по странам

Важно не смешивать `ALL` с реальными странами. `ALL` — агрегат All Countries, а не страна.

Задачи:

- Построить `received` по странам за весь период.
- Построить top-10 стран по `received`.
- Построить top-10 стран по `approved`.
- Построить top-10 стран по `denied`.
- Построить top-10 стран по `pending`.
- Для каждой страны посчитать approval rate на decided basis.
- Для каждой страны посчитать denial rate на decided basis.
- Для каждой страны посчитать pending share.
- Построить график `received` по годам для крупнейших стран.
- Построить график NIW `received` по странам.
- Построить график EB1 `received` по странам.
- Построить график EB2 `received` по странам.
- Построить график EB3 `received` по странам.
- Сравнить структуру категорий по странам: доля EB1, EB2, EB3.
- Сравнить структуру EB2 по странам: E21 vs NIW.
- Найти страны, где NIW занимает максимальную долю среди EB2.
- Найти страны, где denial rate по NIW выше среднего.
- Найти страны, где pending share по NIW выше среднего.
- Сравнить ALL country с weighted average по известным странам и проверить, не означает ли расхождение неполное страновое покрытие.

## Этап 6. Snapshot analysis: как меняются старые когорты в новых отчетах

Ключевые поля:

- `cohort_fiscal_year`
- `cohort_quarter`
- `snapshot_date`
- `report_fiscal_year`
- `report_quarter`
- `status`
- `count_value`

Задачи:

- Для NIW взять конкретную когорту, например FY2024 Q1, и построить таблицу изменения `approved`, `denied`, `pending` по `snapshot_date`.
- Повторить для FY2024 Q2-Q4.
- Повторить для FY2023 Q1-Q4.
- Построить line chart pending count одной когорты по `snapshot_date`.
- Построить line chart approved count одной когорты по `snapshot_date`.
- Построить line chart denied count одной когорты по `snapshot_date`.
- Для каждой когорты посчитать pending decay: `pending_current_snapshot - pending_previous_snapshot`.
- Для каждой когорты посчитать approved increase.
- Для каждой когорты посчитать denied increase.
- Проверить гипотезу: если pending уменьшился на X, то approved + denied должны увеличиться примерно на X.
- Найти когорты, где эта арифметика не сходится.
- Посчитать возраст snapshot: сколько кварталов прошло между cohort period и snapshot date.
- Построить average pending share by cohort age.
- Сделать это отдельно для EB1, EB2, NIW, EB3.
- Сделать это отдельно по странам, если данных достаточно.
- Построить cohort aging heatmap: cohort quarter x snapshot age, значение pending share.
- Построить cohort aging heatmap для cumulative approval share.
- Построить cohort aging heatmap для cumulative denial share.
- Найти, через сколько кварталов pending share обычно падает ниже 50%, 25%, 10%.

## Этап 7. Pending conversion

Цель этапа — оценить, во что исторически превращался pending.

Задачи:

- Для каждой когорты и пары соседних snapshots посчитать `pending_delta = pending_t - pending_t+1`.
- Посчитать `approved_delta = approved_t+1 - approved_t`.
- Посчитать `denied_delta = denied_t+1 - denied_t`.
- Посчитать implied conversion rates: `approved_delta / pending_t`, `denied_delta / pending_t`, `pending_t+1 / pending_t`.
- Посчитать conversion от resolved pending: `approved_delta / (approved_delta + denied_delta)` и `denied_delta / (approved_delta + denied_delta)`.
- Сравнить pending conversion rates по возрасту когорты.
- Сравнить pending conversion rates по категориям: EB1, EB2, NIW, EB3.
- Сравнить NIW pending conversion с E21.
- Сравнить E11 pending conversion с E12/E13.
- Сравнить pending conversion по странам, если выборка достаточно большая.
- Посчитать среднюю долю pending, которая становится denied за следующие 1, 2, 3, 4 квартала.
- Посчитать среднюю долю pending, которая становится approved за следующие 1, 2, 3, 4 квартала.
- Построить график: cohort age -> probability pending resolves into approval.
- Построить график: cohort age -> probability pending resolves into denial.
- Построить график: cohort age -> probability remains pending.
- Отдельно посчитать это для старых, почти полностью дозревших когорт.
- Сравнить historical pending conversion до и после 2023.

## Этап 8. Прогноз итогового результата для свежих когорт

Цель этапа — построить осторожный прогноз итогового approval/denial rate для свежих когорт с большим pending.

Задачи:

- Взять свежую когорту, например FY2025 Q4 NIW.
- Зафиксировать текущие `received`, `approved`, `denied`, `pending`.
- Найти исторические когорты того же возраста, для которых уже видно дальнейшее дозревание.
- Посчитать историческую долю pending, которая стала approved.
- Посчитать историческую долю pending, которая стала denied.
- Посчитать историческую долю pending, которая осталась pending через 1-6 кварталов.
- Построить naive forecast: `expected_final_approved = current_approved + pending * historical_pending_to_approved_rate`.
- Построить naive forecast: `expected_final_denied = current_denied + pending * historical_pending_to_denied_rate`.
- Посчитать `expected_final_approval_rate = expected_final_approved / received`.
- Посчитать `expected_final_denial_rate = expected_final_denied / received`.
- Посчитать доверительные интервалы для pending_to_approved_rate и pending_to_denied_rate.
- Посчитать доверительный интервал для expected final approval rate.
- Сделать optimistic, base и pessimistic сценарии.
- Повторить прогноз для FY2025 Q4 по EB1, EB2, NIW, EB3.
- Повторить прогноз для FY2025 Q1-Q4.
- Проверить, как меняется прогноз, если брать только исторические когорты той же категории.
- Проверить, как меняется прогноз, если брать только последние 2-3 года.
- Проверить, как меняется прогноз при исключении COVID/post-COVID аномальных периодов, если они видны.
- Сделать backtesting на старых когортах.
- Посчитать среднюю ошибку прогноза по approval rate.
- Посчитать среднюю ошибку прогноза по denial rate.
- Проверить, в каком проценте случаев фактический итог попал в доверительный интервал.

## Этап 9. Качество данных и sanity checks

Цель этапа — не дать красивым графикам спрятать проблемы в данных.

Задачи:

- Проверить, что `received` для одной когорты не меняется между snapshots или меняется редко.
- Проверить, что `approved` по одной когорте не уменьшается между snapshots.
- Проверить, что `denied` по одной когорте не уменьшается между snapshots.
- Проверить, что pending в целом уменьшается с возрастом когорты.
- Найти когорты, где pending растет между snapshots.
- Найти когорты, где `approved + denied + pending` сильно отличается от `received`.
- Проверить отдельно `pending` и `pending_other`, чтобы понять, где их можно объединять.
- Проверить, не дублируются ли данные между `fy_quarter_status` и `preference_country`.
- Проверить, отличается ли ALL country от суммы известных стран.
- Проверить consistency между `form_status_counts` для I-140 и агрегатами из `i140_status_counts` по TOTAL / ALL.
- Найти source files с низким `extraction_confidence`.
- Найти rows с `reviewed = false` и большими `count_value`.
- Найти факты, где `raw_row_label` или `raw_column_label` выглядят подозрительно: пустые, слишком длинные, содержат footnotes или странные переносы.

## Этап 10. Визуализации и дашборд

Цель этапа — собрать визуальный слой для регулярного мониторинга и итогового заключения.

Планируемые элементы:

- Dashboard card: total I-140 received за выбранный год.
- Dashboard card: approval rate на decided basis за выбранный год.
- Dashboard card: pending share за выбранный год.
- Line chart: I-140 received by quarter.
- Line chart: approved / denied / pending by quarter.
- Line chart: NIW received by quarter.
- Line chart: EB1A / E11 received by quarter или by year.
- Stacked bar: EB1 / EB2 / EB3 received by year.
- Stacked bar: NIW vs E21 внутри EB2.
- Stacked bar: E11 / E12 / E13 внутри EB1.
- Country bar chart: top countries by received.
- Country heatmap: country x category, значение received.
- Cohort heatmap: cohort quarter x snapshot age, значение pending share.
- Cohort line chart: selected cohort pending decay.
- Forecast table: selected fresh cohort, current observed approval/denial/pending, predicted final approval/denial.
- Confidence interval chart: predicted final approval rate with lower/upper bound.

## Приоритет выполнения

Для ручной pandas-практики сначала лучше выполнить этапы 1-5 и часть этапа 10. Там много группировок, pivot tables, merge, datetime, line plots, stacked bars и heatmaps.

Для более осторожной методологической работы лучше отдельно вынести этапы 6-9: cohort age, transition rates, censoring, backtesting, confidence intervals. Там легко получить красивый, но ложный результат, поэтому эти расчеты нужно делать после первичного понимания данных.

## Принцип обновления проекта

По мере появления новых USCIS данных нужно:

1. Добавить новые исходные файлы в локальную raw-папку.
2. Перезапустить парсер и загрузку в PostgreSQL.
3. Обновить CSV-export.
4. Перезапустить notebooks/дашборд.
5. Проверить coverage и sanity checks.
6. Пересчитать rates, cohort analysis и forecast.
7. Обновить итоговое заключение.

